# Spotify Song Recommendation And Popularity Modeling

This notebook builds the modeling layer for the composer dashboard. It uses `spotify_cleaned_eda.csv` and creates three outputs:

1. Similar existing songs based on audio-feature similarity.
2. Fame-free base popularity prediction using composition features only.
3. Artist-fit recommendations using style similarity plus estimated artist lift.

The workflow now compares several popularity models before selecting the final base model.

## 1. Setup

Use the project virtual environment if possible: `.venv\Scripts\python.exe`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "spotify_train_models.py").exists())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import json
import joblib
import numpy as np
import pandas as pd

from spotify_train_models import (
    BASE_POPULARITY_MODEL_NAME,
    DATA_PATH,
    MODEL_DIR,
    NUMERIC_FEATURES,
    POPULARITY_BANDS,
    POPULARITY_FEATURES,
    RECOMMENDER_FEATURES,
    build_artist_profiles,
    build_model_metadata,
    build_recommender,
    evaluate_popularity_models,
    load_modeling_data,
    predict_base_popularity,
    recommend_artists,
    recommend_songs,
    train_final_popularity_model,
)

## 2. Load Modeling Data

This starts from the cleaned EDA dataset, then applies final modeling checks: required columns, numeric coercion, valid tempo/duration, and non-missing metadata.

In [ ]:
df = load_modeling_data(DATA_PATH)

print(f"Rows after modeling checks: {len(df):,}")
print(f"Unique tracks: {df['track_id'].nunique():,}")
print(f"Genres: {df['track_genre'].nunique():,}")
df.head()

## 3. Feature Design

The base popularity model is intentionally fame-free. It excludes `artists`, `album_name`, `track_name`, and `track_id`. Artist influence is handled later as a separate fit/lift layer.

In [ ]:
print("Recommender features:")
print(RECOMMENDER_FEATURES)

print("Popularity model features:")
print(POPULARITY_FEATURES)

print("Popularity bands:")
print(pd.DataFrame(POPULARITY_BANDS))

## 4. Similar Song Recommender

This model uses scaled audio features and cosine distance. If the user selects a genre, the recommendation helper searches inside that genre first.

In [ ]:
recommender_scaler, recommender, song_catalog = build_recommender(df)

print(f"Song catalog rows after track_id deduplication: {len(song_catalog):,}")
song_catalog.head()

## 5. Popularity Modeling Strategy

Popularity is a social and commercial outcome, so audio features alone should not be expected to explain all variance. We compare several models, then use the best practical baseline for the composer-facing fame-free score.

In [ ]:
model_comparison = evaluate_popularity_models(df)
model_comparison[["model", "mae", "rmse", "r2", "train_rows", "test_rows"]].round(4)

## 6. Select Final Base Popularity Model

The selected model is the tuned histogram gradient boosting regressor. It performed best in the comparison split while keeping the input features fame-free.

In [ ]:
selected_model_name = BASE_POPULARITY_MODEL_NAME
selected_metrics = model_comparison[model_comparison["model"] == selected_model_name].iloc[0]

print(f"Selected model: {selected_model_name}")
print(selected_metrics[["mae", "rmse", "r2", "unique_train_tracks", "unique_test_tracks"]])

## 7. Train Final Base Popularity Model

After evaluation, retrain the selected model on the full modeling dataset so the dashboard uses all available training data.

In [ ]:
final_popularity_model = train_final_popularity_model(df, selected_model_name)
print("Final fame-free base popularity model trained.")

## 8. Artist Fit Profiles

Artist fit is estimated separately from base popularity. The model calculates historical artist residual lift: actual popularity minus the base composition-only prediction. This avoids contaminating the base composer score with artist fame.

In [ ]:
artist_profiles = build_artist_profiles(df, final_popularity_model, min_tracks=5)

print(f"Artist profiles: {len(artist_profiles):,}")
artist_profiles.head(10)

## 9. Example Composer Input

This simulates the dashboard inputs. Change these values to test different prospective songs.

In [ ]:
composer_input = {
    "danceability": 0.70,
    "energy": 0.75,
    "loudness": -6.5,
    "speechiness": 0.06,
    "acousticness": 0.18,
    "instrumentalness": 0.00,
    "liveness": 0.12,
    "valence": 0.62,
    "tempo": 120.0,
    "duration_min": 3.4,
    "explicit_int": 0,
    "key": 1,
    "mode": 1,
    "time_signature": 4,
    "track_genre": "pop",
}

composer_df = pd.DataFrame([composer_input])
composer_df

## 10. Output 1: Similar Existing Songs

In [ ]:
similar_songs = recommend_songs(
    input_profile=composer_input,
    scaler=recommender_scaler,
    recommender=recommender,
    song_catalog=song_catalog,
    top_n=10,
    genre=composer_input["track_genre"],
)

similar_songs[[
    "track_name", "artists", "track_genre", "popularity", "similarity_score",
    "danceability", "energy", "valence", "tempo"
]]

## 11. Output 2: Base Popularity Prediction

This is the fame-free estimate. In the dashboard, the band is usually more reliable than over-interpreting exact score differences.

In [ ]:
base_popularity_prediction, popularity_label = predict_base_popularity(composer_input, final_popularity_model)

print(f"Base popularity prediction: {base_popularity_prediction:.2f} / 100")
print(f"Popularity band: {popularity_label}")

## 12. Output 3: Recommended Artists To Pitch To

The artist-assisted score combines base popularity, artist residual lift, and style similarity. Treat it as a decision-support estimate, not a guarantee of virality.

In [ ]:
artist_recommendations = recommend_artists(
    input_profile=composer_input,
    artist_profiles=artist_profiles,
    base_popularity_prediction=base_popularity_prediction,
    scaler=recommender_scaler,
    top_n=10,
    genre=composer_input["track_genre"],
)

artist_recommendations

## 13. Save Dashboard Artifacts

These files are what the Streamlit dashboard should load later.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(recommender_scaler, MODEL_DIR / "spotify_recommender_scaler.joblib")
joblib.dump(recommender, MODEL_DIR / "spotify_nearest_neighbors.joblib")
joblib.dump(final_popularity_model, MODEL_DIR / "spotify_base_popularity_model.joblib")
song_catalog.to_csv(MODEL_DIR / "spotify_song_catalog.csv", index=False)
artist_profiles.to_csv(MODEL_DIR / "spotify_artist_profiles.csv", index=False)
model_comparison.to_csv(MODEL_DIR / "spotify_popularity_model_comparison.csv", index=False)

metadata = build_model_metadata(
    df=df,
    song_catalog=song_catalog,
    artist_profiles=artist_profiles,
    model_comparison=model_comparison,
    selected_model_name=selected_model_name,
)
(MODEL_DIR / "spotify_model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved model artifacts to: {MODEL_DIR}")

## 14. Modeling Interpretation

The selected base model is useful for a prototype, but the score must be positioned correctly. A moderate `R?` is expected because popularity depends on many hidden social and commercial variables. For the dashboard, use the exact score as an estimate and the Low/Moderate/High band as the more stable user-facing interpretation. The artist-fit layer should remain separate from the base score so the composer can distinguish song potential from artist amplification.